# Clase 6 · Notebook 02
# Tu primera CNN con Keras

**Introducción al Deep Learning — Módulo 3, Unidad 1: Redes de neuronas convolucionales**

Construiremos una **red de neuronas convolucional (CNN)** completa para clasificar imágenes de dígitos
manuscritos. La red tendrá las dos secciones que vimos en teoría:

- **Extracción**: capas `Conv2D` + `MaxPooling2D` + `Flatten`.
- **Clasificación**: capas `Dense` (la última con `softmax`).

Usamos el dataset `digits` (8×8) para que entrene rápido y sin descargas.

> 💡 En Colab, TensorFlow ya viene instalado. Ejecuta las celdas en orden con **Shift + Enter**.

## 1. Preparar las imágenes

In [ ]:
try:
    import tensorflow as tf
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow"])
    import tensorflow as tf
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
tf.random.set_seed(42); np.random.seed(42)

digits = load_digits()
# Una CNN espera imágenes con forma (alto, ancho, canales). Aquí: 8x8x1 (un canal, escala de grises)
X = digits.images.reshape(-1, 8, 8, 1).astype("float32") / 16.0   # normalizamos 0-1
y = tf.keras.utils.to_categorical(digits.target, num_classes=10)  # 10 dígitos (one-hot)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=digits.target)
print("Imágenes de entrenamiento:", X_train.shape, "| test:", X_test.shape)

## 2. Construir la CNN

- **Conv2D** (8 filtros 3×3, ReLU): extrae características; `padding='same'` conserva el tamaño.
- **MaxPooling2D** (2×2): reduce la dimensión.
- **Flatten**: pasa de matriz a vector.
- **Dense** (ReLU) y **Dense** (10, softmax): clasifican entre los 10 dígitos.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input

modelo = Sequential([
    Input(shape=(8, 8, 1)),
    Conv2D(8, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2)),
    Conv2D(16, (3, 3), activation="relu", padding="same"),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(32, activation="relu"),
    Dense(10, activation="softmax"),
])
modelo.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
modelo.summary()

Observa en el resumen que las capas de **pooling no tienen parámetros** que entrenar, mientras que las
convolucionales y densas sí.

## 3. Entrenar

In [ ]:
historia = modelo.fit(X_train, y_train, epochs=15, batch_size=32,
                      validation_split=0.1, verbose=0)
acc = modelo.evaluate(X_test, y_test, verbose=0)[1]
print("Accuracy en test: %.1f %%" % (acc * 100))

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(historia.history["accuracy"], label="entrenamiento", color="#000F9F")
plt.plot(historia.history["val_accuracy"], label="validación", color="#FF647E")
plt.title("Exactitud durante el entrenamiento"); plt.xlabel("Época"); plt.ylabel("Accuracy")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

## 4. Probar el modelo con imágenes del test

In [ ]:
pred = modelo.predict(X_test, verbose=0)
y_pred = pred.argmax(axis=1)
y_true = y_test.argmax(axis=1)

fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
for k, ax in enumerate(axes):
    ax.imshow(X_test[k].reshape(8, 8), cmap="gray")
    ok = "OK" if y_pred[k] == y_true[k] else "X"
    ax.set_title(f"pred {y_pred[k]} ({ok})", fontsize=10,
                 color=("#00A38C" if y_pred[k]==y_true[k] else "#FF647E"))
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Matriz de confusión (10 dígitos)"); plt.tight_layout(); plt.show()

## Experimenta tú

1. Añade una tercera capa `Conv2D` o sube el número de filtros: ¿mejora la accuracy?
2. Quita el `MaxPooling2D`: ¿cambia el número de parámetros y el tiempo?
3. Cambia `padding='same'` por `'valid'` y observa los tamaños en `summary()`.

## Resumen

- Una **CNN** combina **Conv2D** (extrae características) + **MaxPooling2D** (reduce) + **Flatten** + **Dense** (clasifica).
- Las imágenes se dan con forma `(alto, ancho, canales)` y se normalizan.
- La red **aprende sola** los filtros: no hace falta diseñarlos a mano.
- Salida `softmax` + pérdida `categorical_crossentropy` para clasificación multiclase.

Has construido tu primer clasificador de imágenes. En la próxima clase veremos cómo entrenar redes aún más
profundas con **conexiones residuales**.